# 客户流失预测 - 模型训练

In [1]:
import os
import pandas as pd
import numpy as np
import dask.dataframe as dd
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.impute import KNNImputer
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import mutual_info_classif
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from category_encoders import TargetEncoder
import warnings

warnings.filterwarnings('ignore')

os.environ["MKL_NUM_THREADS"] = "14"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMBA_NUM_THREADS"] = "14"
os.environ["XGBOOST_THREAD"] = "1"

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
def feature_engineering_dask_fast(train, test):
    df_train = dd.from_pandas(train.copy(), npartitions=14)
    df_test = dd.from_pandas(test.copy(), npartitions=14)
    y_train = train['Exited'].values

    for df in [df_train, df_test]:
        df['Surname'] = df['Surname'].fillna('Unknown')
        df['Surname_Len'] = df['Surname'].str.len()
        df['AgeWhenJoin'] = df['Age'] - df['Tenure']
        df['BalanceZero'] = (df['Balance'] == 0).astype(int)
        df['Is3Prod'] = (df['NumOfProducts'] == 3).astype(int)
        df['Is4Prod'] = (df['NumOfProducts'] == 4).astype(int)
        df['InactiveNoBalance'] = (df['IsActiveMember'] == 0) & (df['Balance'] == 0)
        df['BalSalaryRatio'] = df['Balance'] / (df['EstimatedSalary'] + 1e-8)
        df['AgePerProd'] = df['Age'] / (df['NumOfProducts'] + 1)
        df['BalPerTenure'] = df['Balance'] / (df['Tenure'] + 1)
        df['AgeTenureRatio'] = df['Age'] / (df['Tenure'] + 1)
        df['SalaryPerProd'] = df['EstimatedSalary'] / (df['NumOfProducts'] + 1)
        df['AgeBalRatio'] = df['Age'] / (df['Balance'] + 1e-8)
        df['CreditAge'] = df['CreditScore'] * df['Age']
        df['TenureBal'] = df['Tenure'] * df['Balance']
        df['ActiveProd'] = df['IsActiveMember'] * df['NumOfProducts']
        df['AgeProdActive'] = df['Age'] * df['NumOfProducts'] * df['IsActiveMember']
        df['CreditBal'] = df['CreditScore'] * df['Balance']
        df['TenureAgeBal'] = df['Tenure'] * df['Age'] * df['Balance']
        df['LogBal'] = np.log1p(df['Balance'])
        df['LogSalary'] = np.log1p(df['EstimatedSalary'])
        df['LogBalSalary'] = np.log1p(df['BalSalaryRatio'])
        df['AgeBin'] = df['Age'].map(lambda x: np.digitize(x, bins=[0,25,35,45,55,65,100])-1)
        df['CreditBin'] = df['CreditScore'].map(lambda x: np.digitize(x, bins=[0,400,500,600,700,800,1000])-1)
        for c in ['Age', 'Balance', 'CreditScore', 'EstimatedSalary']:
            q25 = df[c].quantile(0.25)
            q75 = df[c].quantile(0.75)
            df[f'{c}_q25'] = (df[c] < q25).astype(int)
            df[f'{c}_q75'] = (df[c] > q75).astype(int)
        df['HighAgeLowCredit'] = ((df['Age'] > 45) & (df['CreditScore'] < 550)).astype(int)
        df['InactiveHighBal'] = ((df['IsActiveMember'] == 0) & (df['Balance'] > 50000)).astype(int)
        df['ManyProdLowTenure'] = ((df['NumOfProducts'] >= 3) & (df['Tenure'] <= 2)).astype(int)
        df['Geo_Gender'] = df['Geography'] + "_" + df['Gender']
        df['Geo_Prod'] = df['Geography'].astype(str) + "_" + df['NumOfProducts'].astype(str)

    X_train = df_train.compute()
    X_test = df_test.compute()

    le_sur = LabelEncoder()
    X_train['Surname'] = le_sur.fit_transform(X_train['Surname'])
    X_test['Surname'] = X_test['Surname'].apply(lambda x: le_sur.transform([x])[0] if x in le_sur.classes_ else len(le_sur.classes_))

    te_cols = ['Geography', 'Gender', 'Geo_Gender', 'Geo_Prod', 'AgeBin', 'NumOfProducts', 'Surname']
    te = TargetEncoder(cols=te_cols, smoothing=5)
    X_train = pd.concat([X_train, te.fit_transform(X_train[te_cols], y_train).add_suffix("_te")], axis=1)
    X_test = pd.concat([X_test, te.transform(X_test[te_cols]).add_suffix("_te")], axis=1)

    for c in ['Age', 'CreditScore', 'Balance', 'EstimatedSalary']:
        X_train[f'{c}_bin'], b = pd.qcut(X_train[c], 5, duplicates='drop', retbins=True)
        X_test[f'{c}_bin'] = pd.cut(X_test[c], b, labels=False)
        g = X_train.groupby(f'{c}_bin')['Exited'].agg(pos='sum', cnt='count')
        g['neg'] = g['cnt'] - g['pos']
        pos_all = y_train.sum()
        neg_all = len(y_train) - pos_all
        g['woe'] = np.log((g['pos'] / pos_all + 1e-8) / (g['neg'] / neg_all + 1e-8))
        X_train[f'{c}_woe'] = X_train[f'{c}_bin'].map(g['woe'])
        X_test[f'{c}_woe'] = X_test[f'{c}_bin'].map(g['woe']).fillna(0)

    drop = ['RowNumber', 'CustomerId', 'id', 'Exited'] + [c for c in X_train.columns if '_bin' in c]
    X_train = X_train.drop(columns=[c for c in drop if c in X_train.columns], errors='ignore')
    X_test = X_test.drop(columns=[c for c in drop if c in X_test.columns], errors='ignore')

    for c in ['Geography', 'Gender', 'Geo_Gender', 'Geo_Prod']:
        le = LabelEncoder()
        X_train[c] = le.fit_transform(X_train[c])
        X_test[c] = le.transform(X_test[c])

    return X_train, X_test, y_train

In [3]:
class FastResMLP(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, 128),
            nn.BatchNorm1d(128),
            nn.Mish(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

def train_mlp_fast(xtr, ytr, xval, yval, input_dim):
    model = FastResMLP(input_dim).to(DEVICE)
    pos = torch.tensor([(ytr == 0).sum() / (ytr == 1).sum()], dtype=torch.float32).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos)
    opt = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    Xtr = torch.tensor(xtr, dtype=torch.float32).to(DEVICE)
    Ytr = torch.tensor(ytr, dtype=torch.float32).to(DEVICE)
    ds = TensorDataset(Xtr, Ytr)
    loader = DataLoader(ds, batch_size=512, shuffle=True, num_workers=0)

    Xval = torch.tensor(xval, dtype=torch.float32).to(DEVICE)
    Yval = torch.tensor(yval, dtype=torch.float32).to(DEVICE)

    best, pat, cnt = 1e9, 8, 0
    for _ in range(50):
        model.train()
        for bx, by in loader:
            opt.zero_grad()
            criterion(model(bx), by).backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            vloss = criterion(model(Xval), Yval).item()

        if vloss < best:
            best, cnt = vloss, 0
        else:
            cnt += 1
            if cnt >= pat:
                break
    return model

def pseudo_fast(X, y, X_te):
    m = lgb.LGBMClassifier(n_estimators=1200, learning_rate=0.025, max_depth=6,
                           reg_alpha=0.2, reg_lambda=0.2, device='cuda', n_jobs=14, random_state=SEED, verbose=-1)
    m.fit(X, y, eval_set=[(X, y)], callbacks=[lgb.early_stopping(20, verbose=0)])
    prob = m.predict_proba(X_te)[:, 1]
    mask = (prob > 0.89) | (prob < 0.11)
    idx = np.where(mask)[0]
    return X_te[idx], (prob[idx] > 0.5).astype(int)

def train_gold_stack_fast(X_full, y_full, X_test, cat_features):
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
    oof = np.zeros((len(X_full), 4))
    pred = np.zeros((len(X_test), 4))

    scaler = StandardScaler()
    X_s = scaler.fit_transform(X_full)
    Xt_s = scaler.transform(X_test)

    lgb_p = {'learning_rate': 0.012, 'num_leaves': 192, 'max_depth': 10, 'reg_alpha': 0.2, 'reg_lambda': 0.5, 'subsample': 0.95, 'colsample_bytree': 0.95}
    xgb_p = {'learning_rate': 0.012, 'max_depth': 9, 'reg_alpha': 0.2, 'reg_lambda': 0.5, 'subsample': 0.95, 'colsample_bytree': 0.95}
    cat_p = {'learning_rate': 0.014, 'depth': 8, 'l2_leaf_reg': 2.0, 'subsample': 0.95, 'bootstrap_type': 'Bernoulli'}

    for fold, (tr, val) in enumerate(skf.split(X_full, y_full)):
        xt, xv = X_full[tr], X_full[val]
        yt, yv = y_full[tr], y_full[val]
        xs_t, xs_v = X_s[tr], X_s[val]

        m1 = lgb.LGBMClassifier(**lgb_p, n_estimators=25000, device='cuda', n_jobs=14, verbose=-1)
        m1.fit(xt, yt, eval_set=[(xv, yv)], callbacks=[lgb.early_stopping(200, verbose=False)])

        m2 = xgb.XGBClassifier(**xgb_p, n_estimators=25000, tree_method='hist', device='cuda', nthread=1, verbosity=0, early_stopping_rounds=200)
        m2.fit(xt, yt, eval_set=[(xv, yv)], verbose=False)

        m3 = CatBoostClassifier(**cat_p, iterations=25000, task_type='GPU', verbose=False, early_stopping_rounds=200)
        m3.fit(xt, yt, eval_set=(xv, yv), use_best_model=True)

        m7 = train_mlp_fast(xs_t, yt, xs_v, yv, X_s.shape[1])

        oof[val, 0] = m1.predict_proba(xv)[:, 1]
        oof[val, 1] = m2.predict_proba(xv)[:, 1]
        oof[val, 2] = m3.predict_proba(xv)[:, 1]
        with torch.no_grad():
            oof[val, 3] = torch.sigmoid(m7(torch.tensor(xs_v, dtype=torch.float32).to(DEVICE))).cpu().numpy()

        pred[:, 0] += m1.predict_proba(X_test)[:, 1] / 10
        pred[:, 1] += m2.predict_proba(X_test)[:, 1] / 10
        pred[:, 2] += m3.predict_proba(X_test)[:, 1] / 10
        with torch.no_grad():
            pred[:, 3] += torch.sigmoid(m7(torch.tensor(Xt_s, dtype=torch.float32).to(DEVICE))).cpu().numpy() / 10

    meta = LogisticRegression(C=0.05, solver='liblinear', max_iter=2000, random_state=SEED)
    meta.fit(oof, y_full)
    final = meta.predict_proba(pred)[:, 1]
    return final

In [4]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
train.columns = train.columns.str.strip()
test.columns = test.columns.str.strip()

X, Xt, y = feature_engineering_dask_fast(train, test)

imp = KNNImputer(n_neighbors=8)
X = imp.fit_transform(X)
Xt = imp.transform(Xt)

mi = mutual_info_classif(X, y, random_state=SEED)
mask = mi > np.percentile(mi, 20)
X_selected = X[:, mask]
Xt_selected = Xt[:, mask]

X_p, y_p = pseudo_fast(X_selected, y, Xt_selected)
X_full = np.vstack([X_selected, X_p])
y_full = np.hstack([y, y_p])

final_pred = train_gold_stack_fast(X_full, y_full, Xt_selected, None)

sub = pd.DataFrame({"id": test.id, "Exited": final_pred})
sub.to_csv("submission.csv", index=False)

KeyboardInterrupt: 